# 02 - Baseline Models And Submission

This notebook builds reproducible starter baselines.

Modeling choices here are deliberately conservative:
- exclude `ticker` from the first baseline
- use only features available at the observation date
- evaluate on a 2022 holdout before fitting on all training rows

The goal is not leaderboard magic. The goal is a clean baseline that the team can improve.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['period_start', 'period_end'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['period_start', 'period_end'])
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

train.shape, test.shape

In [ ]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame.copy()

    features['obs_year'] = features['period_start'].dt.year
    features['obs_quarter'] = features['period_start'].dt.quarter
    features['obs_month'] = features['period_start'].dt.month

    drop_columns = ['id', 'ticker', 'period_start', 'period_end', 'return_pct']
    existing_drop_columns = [col for col in drop_columns if col in features.columns]
    features = features.drop(columns=existing_drop_columns)

    numeric_columns = features.select_dtypes(include=[np.number]).columns.tolist()
    return features[numeric_columns]


X_full = build_features(train)
X_test = build_features(test)
y_full = train['return_pct']

print('Number of modeling features:', X_full.shape[1])
display(X_full.head())

In [ ]:
train_mask = train['period_start'] < '2022-01-01'
valid_mask = ~train_mask

X_train = X_full.loc[train_mask]
y_train = y_full.loc[train_mask]
X_valid = X_full.loc[valid_mask]
y_valid = y_full.loc[valid_mask]

print('X_train shape:', X_train.shape)
print('X_valid shape:', X_valid.shape)

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

ridge_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        (
            'model',
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_depth=4,
                max_iter=300,
                min_samples_leaf=40,
                random_state=42,
            ),
        ),
    ]
)

models = {
    'dummy_mean': DummyRegressor(strategy='mean'),
    'dummy_median': DummyRegressor(strategy='median'),
    'ridge': ridge_pipeline,
    'hist_gradient_boosting': hgb_pipeline,
}

scores = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = rmse(y_valid, preds)
    scores.append({'model': name, 'rmse': score})
    fitted_models[name] = model

scores = pd.DataFrame(scores).sort_values('rmse')
display(scores)

In [ ]:
best_model_name = scores.iloc[0]['model']
best_model = fitted_models[best_model_name]

valid_predictions = best_model.predict(X_valid)
residuals = pd.DataFrame(
    {
        'ticker': train.loc[valid_mask, 'ticker'].values,
        'period_start': train.loc[valid_mask, 'period_start'].values,
        'actual': y_valid.values,
        'predicted': valid_predictions,
    }
)
residuals['abs_error'] = (residuals['actual'] - residuals['predicted']).abs()

print('Best validation model:', best_model_name)
display(residuals.sort_values('abs_error', ascending=False).head(15))

## Refit On All Training Data

After selecting a baseline model using the holdout, refit it on the full training set before generating the Kaggle submission.

In [ ]:
final_model = models[best_model_name]
final_model.fit(X_full, y_full)
test_predictions = final_model.predict(X_test)

submission = sample_submission.copy()
submission['return_pct'] = test_predictions

submission_path = SUBMISSION_DIR / f'{best_model_name}_baseline.csv'
submission.to_csv(submission_path, index=False)

print('Saved submission to:', submission_path)
display(submission.head())
print('Submission rows:', len(submission))

## Next Experiments

Once this notebook runs cleanly, the next sensible experiments are:

- target clipping or winsorization on the training fold only
- sector-relative features
- log transforms for scale-heavy accounting variables
- LightGBM or XGBoost with the same time-based validation split
- model blending once two models show independent value